# Parte I: GLM Poisson — Estimación por Máxima Verosimilitud

Este notebook implementa el GLM Poisson desde cero sobre datos simulados,
donde el verdadero valor de los parámetros es conocido. El objetivo es verificar
que la teoría funciona cuando los datos se portan bien — antes de enfrentar
datos reales con sobredispersión.

Contenido:
1. Formulación del GLM Poisson
2. Simulación de datos
3. MLE manual con `scipy.optimize`
4. Verificación con `statsmodels`
5. Diagnóstico de sobredispersión

## 1. El modelo GLM Poisson

### Formulación

$$Y_i \sim \text{Poisson}(\lambda_i), \quad i = 1, \ldots, n$$

$$\log(\lambda_i) = \eta_i = \mathbf{x}_i^\top \boldsymbol{\beta}$$

donde $\mathbf{x}_i = (1, x_{i1}, \ldots, x_{ip})^\top$ y
$\boldsymbol{\beta} = (\beta_0, \beta_1, \ldots, \beta_p)^\top$.

La función de enlace logarítmica garantiza $\lambda_i > 0$ para cualquier valor de $\eta_i$.

### Log-verosimilitud

La función de masa de probabilidad de Poisson es:

$$P(Y_i = y_i \mid \lambda_i) = \frac{e^{-\lambda_i} \lambda_i^{y_i}}{y_i!}$$

La log-verosimilitud del vector de observaciones $(y_1, \ldots, y_n)$ es:

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^n \left[
  y_i \cdot \mathbf{x}_i^\top \boldsymbol{\beta}
  - e^{\mathbf{x}_i^\top \boldsymbol{\beta}}
  - \log(y_i!)
\right]$$

Como $\log(y_i!)$ no depende de $\boldsymbol{\beta}$, maximizamos la forma reducida:

$$\ell(\boldsymbol{\beta}) \propto \sum_{i=1}^n \left[
  y_i \cdot \mathbf{x}_i^\top \boldsymbol{\beta}
  - e^{\mathbf{x}_i^\top \boldsymbol{\beta}}
\right]$$

### Gradiente y Hessiana

Definiendo $\mu_i = e^{\mathbf{x}_i^\top \boldsymbol{\beta}}$ (media condicional):

**Gradiente** (score):

$$\nabla_{\boldsymbol{\beta}} \ell = \mathbf{X}^\top (\mathbf{y} - \boldsymbol{\mu})$$

donde $\boldsymbol{\mu} = (\mu_1, \ldots, \mu_n)^\top$.

**Hessiana** (negativa definida en el máximo):

$$\mathbf{H} = -\mathbf{X}^\top \mathbf{W} \mathbf{X}, \quad \mathbf{W} = \text{diag}(\mu_1, \ldots, \mu_n)$$

**Covarianza asintótica** del estimador MLE:

$$\text{Cov}(\hat{\boldsymbol{\beta}}) \approx (-\mathbf{H})^{-1} = (\mathbf{X}^\top \mathbf{W} \mathbf{X})^{-1}$$

Esta es la **Información de Fisher Observada** evaluada en $\hat{\boldsymbol{\beta}}$.

*[Espacio para ampliar: derivación del gradiente por regla de la cadena, condiciones de segundo orden, convergencia de Newton-Raphson...]*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from scipy.optimize import minimize

sns.set_style('darkgrid')
np.random.seed(44)

## 2. Simulación de datos

Generamos datos donde el verdadero modelo es Poisson con parámetros conocidos.
Usamos $\log(x)$ como predictor para capturar relaciones multiplicativas.

Modelo verdadero:

$$\log(\lambda_i) = \beta_0 + \beta_1 \log(x_i), \quad \beta_0 = 1.0, \quad \beta_1 = 0.8$$

In [ ]:
n = 1500
x = np.random.normal(loc=35, scale=6, size=n)
log_x = np.log(x)

# Parámetros verdaderos
beta_0_real, beta_1_real = 1.0, 0.8

lam = np.exp(beta_0_real + beta_1_real * log_x)
y = np.random.poisson(lam=lam, size=n)

df_sim = pd.DataFrame({'x': x, 'log_x': log_x, 'y': y})

print(f'n = {n}')
print(f'Parámetros reales: β₀ = {beta_0_real}, β₁ = {beta_1_real}')
print(f'Media de y: {y.mean():.3f}  |  Varianza de y: {y.var():.3f}')
print(f'Índice de dispersión D = {y.var()/y.mean():.3f}  (esperado ≈ 1.0 para Poisson puro)')

### Verificación del supuesto Var[Y] ≈ E[Y]

En datos verdaderamente Poisson, la varianza condicional debe aproximarse a la media.

In [ ]:
df_sim['x_cat'] = pd.qcut(df_sim['x'], q=10)
grupo = df_sim.groupby('x_cat', observed=False)['y'].agg(
    n='count', media='mean', varianza='var'
).round(3)
grupo['D'] = (grupo['varianza'] / grupo['media']).round(3)
print(grupo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Hexbin: relación X vs Y
hb = axes[0].hexbin(df_sim['x'], df_sim['y'], gridsize=25, cmap='viridis', mincnt=1)
plt.colorbar(hb, ax=axes[0], label='n observaciones')
axes[0].set(title='Relación X vs Y (densidad hexagonal)', xlabel='X', ylabel='Conteo Y')

# Boxplot por deciles
sns.boxplot(data=df_sim, x='x_cat', y='y', palette='viridis', ax=axes[1])
axes[1].set(title='Distribución de Y por deciles de X', xlabel='Deciles de X', ylabel='Y')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. MLE manual con `scipy.optimize`

Implementamos directamente las tres funciones derivadas de la log-verosimilitud.

In [ ]:
X_mat = np.column_stack([np.ones(n), log_x])

def poisson_nll(beta, X, y):
    mu = np.exp(X @ beta)
    return -np.sum(y * (X @ beta) - mu)

def poisson_gradient(beta, X, y):
    mu = np.exp(X @ beta)
    return -(X.T @ (y - mu))

def poisson_hessian(beta, X, y):
    mu = np.exp(X @ beta)
    return X.T * mu @ X

In [ ]:
beta_init = np.zeros(2)

res = minimize(
    fun=poisson_nll, x0=beta_init, args=(X_mat, y),
    method='Newton-CG', jac=poisson_gradient, hess=poisson_hessian
)

beta_hat = res.x
print('Optimización exitosa:', res.success)
print(f'β̂₀ = {beta_hat[0]:.4f}  (real: {beta_0_real})')
print(f'β̂₁ = {beta_hat[1]:.4f}  (real: {beta_1_real})')

In [ ]:
# Covarianza = inversa de la Hessiana observada
H = poisson_hessian(beta_hat, X_mat, y)
cov = np.linalg.inv(H)
se = np.sqrt(np.diag(cov))
z = beta_hat / se
p_vals = 2 * (1 - stats.norm.cdf(np.abs(z)))
z_crit = stats.norm.ppf(0.975)

print(f'\n{"Variable":<15} {"Coef":>8} {"SE":>8} {"z":>8} {"P>|z|":>8} {"[0.025":>8} {"0.975]":>8}')
print('-' * 65)
for i, name in enumerate(['Intercepto', 'log(x)']):
    print(f'{name:<15} {beta_hat[i]:8.4f} {se[i]:8.4f} {z[i]:8.4f} '
          f'{p_vals[i]:8.4f} {beta_hat[i]-z_crit*se[i]:8.4f} {beta_hat[i]+z_crit*se[i]:8.4f}')

## 4. Verificación con `statsmodels`

Comparamos con la implementación de referencia. Los resultados deben ser idénticos.

In [ ]:
model = smf.poisson(formula='y ~ log_x', data=df_sim)
results = model.fit(disp=0)
print(results.summary())

In [ ]:
# Incident Rate Ratios (IRR): exp(coeficiente)
# Interpretación: por cada unidad que aumenta log(x), y se multiplica por IRR
irr = np.exp(results.params)
print('IRR (exp(coef)):')
print(irr.round(4))

## 5. Diagnóstico de sobredispersión

Para datos simulados de forma verdaderamente Poisson, el índice φ debe ser ≈ 1.
Si φ >> 1, habría sobredispersión y el modelo Poisson sería inadecuado.

$$\hat{\phi} = \frac{\chi^2_{\text{Pearson}}}{n - p}$$

In [ ]:
phi = np.sum(results.resid_pearson**2) / results.df_resid
print(f'Dispersión estimada φ = {phi:.4f}')
if phi > 1.5:
    print('⚠ Sobredispersión detectada (φ >> 1). Considerar Binomial Negativa.')
else:
    print('✓ Dispersión aceptable para Poisson (φ ≈ 1). El supuesto se cumple.')

plt.figure(figsize=(8, 4))
plt.scatter(results.fittedvalues, results.resid_pearson, alpha=0.4)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores ajustados')
plt.ylabel('Residuos de Pearson')
plt.title(f'Residuos vs Ajustados (φ = {phi:.3f})')
plt.tight_layout()
plt.show()

## 6. Conclusiones

Cuando los datos provienen de un proceso verdaderamente Poisson:

- El estimador MLE recupera los parámetros verdaderos con precisión
- Los errores estándar vía Hessiana son correctos y coinciden con `statsmodels`
- El índice de dispersión φ ≈ 1 confirma que el supuesto se cumple

Este es el caso ideal. En los notebooks `01` y `01b` veremos que el dataset real
(Horseshoe Crab Satellites) tiene φ ≈ 3.38 — el modelo Poisson no es adecuado.
El EDA frecuentista motiva los modelos bayesianos que exploramos en los notebooks `03`–`05`.